# Aviation Accidents Analysis

You are part of a consulting firm that is tasked to do an analysis of commercial and passenger jet airline safety. The client (an airline/airplane insurer) is interested in knowing what types of aircraft (makes/models) exhibit low rates of total destruction and low likelihood of fatal or serious passenger injuries in the event of an accident. They are also interested in any general variables/conditions that might be at play. Your analysis will be based off of aviation accident data accumulated from the years 1948-2023. 

Our client is only interested in airplane makes/models that are professional builds and could potentially still be active. Assume a max lifetime of 40 years for a make/model retirement and make sure to filter your data accordingly (i.e. from 1983 onwards). They would also like separate recommendations for small aircraft vs. larger passenger models. **In addition, make sure that claims that you make are statistically robust and that you have enough samples when making comparisons between groups.**


In this summative assessment you will demonstrate your ability to:
- **Use Pandas to load, inspect, and clean the dataset appropriately.**
- **Transform relevant columns to create measures that address the problem at hand.**
- conduct EDA: visualization and statistical measures to systematically understand the structure of the data
- recommend a set of airplanes and makes conforming to the client's request and identify at least *two* factors contributing to airplane safety. You must provide supporting evidence (visuals, summary statistics, tables) for each claim you make.

### Make relevant library imports

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Data Loading and Inspection

### Load in data from the relevant directory and inspect the dataframe.
- inspect NaNs, datatypes, and summary statistics

In [4]:
# Load the dataset into a DataFrame
df = pd.read_csv('data/AviationData.csv', encoding='latin-1', low_memory=False)

# Display the first few rows
df.head()

,Event.Id,Investigation.Type,Accident.Number,Event.Date,Location,Country,Latitude,Longitude,Airport.Code,Airport.Name,...,Purpose.of.flight,Air.carrier,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured,Weather.Condition,Broad.phase.of.flight,Report.Status,Publication.Date
0,20001218X45444,Accident,SEA87LA080,1948-10-24,"MOOSE CREEK, ID",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,2.0,0.0,0.0,0.0,UNK,Cruise,Probable Cause,NaN
1,20001218X45447,Accident,LAX94LA336,1962-07-19,"BRIDGEPORT, CA",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,4.0,0.0,0.0,0.0,UNK,Unknown,Probable Cause,19-09-1996
2,20061025X01555,Accident,NYC07LA005,1974-08-30,"Saltville, VA",United States,36.922223,-81.878056,NaN,NaN,...,Personal,NaN,3.0,NaN,NaN,NaN,IMC,Cruise,Probable Cause,26-02-2007
3,20001218X45448,Accident,LAX96LA321,1977-06-19,"EUREKA, CA",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,2.0,0.0,0.0,0.0,IMC,Cruise,Probable Cause,12-09-2000
4,20041105X01764,Accident,CHI79FA064,1979-08-02,"Canton, OH",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,1.0,2.0,NaN,0.0,VMC,Approach,Probable Cause,16-04-1980


In [5]:
# Check the shape and data types
print(df.shape)
print(df.dtypes)

(88889, 31)
Event.Id                   object
Investigation.Type         object
Accident.Number            object
Event.Date                 object
Location                   object
Country                    object
Latitude                   object
Longitude                  object
Airport.Code               object
Airport.Name               object
Injury.Severity            object
Aircraft.damage            object
Aircraft.Category          object
Registration.Number        object
Make                       object
Model                      object
Amateur.Built              object
Number.of.Engines         float64
Engine.Type                object
FAR.Description            object
Schedule                   object
Purpose.of.flight          object
Air.carrier                object
Total.Fatal.Injuries      float64
Total.Serious.Injuries    float64
Total.Minor.Injuries      float64
Total.Uninjured           float64
Weather.Condition          object
Broad.phase.of.flight      object
Re

In [6]:
# Check missing values per column
df.isnull().sum()

Event.Id                      0
Investigation.Type            0
Accident.Number               0
Event.Date                    0
Location                     52
Country                     226
Latitude                  54507
Longitude                 54516
Airport.Code              38757
Airport.Name              36185
Injury.Severity            1000
Aircraft.damage            3194
Aircraft.Category         56602
Registration.Number        1382
Make                         63
Model                        92
Amateur.Built               102
Number.of.Engines          6084
Engine.Type                7096
FAR.Description           56866
Schedule                  76307
Purpose.of.flight          6192
Air.carrier               72241
Total.Fatal.Injuries      11401
Total.Serious.Injuries    12510
Total.Minor.Injuries      11933
Total.Uninjured            5912
Weather.Condition          4492
Broad.phase.of.flight     27165
Report.Status              6384
Publication.Date          13771
dtype: i

In [7]:
# Summary statistics
df.describe()

,Number.of.Engines,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured
count,82805.000000,77488.000000,76379.000000,76956.000000,82977.000000
mean,1.146585,0.647855,0.279881,0.357061,5.325440
std,0.446510,5.485960,1.544084,2.235625,27.913634
min,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1.000000,0.000000,0.000000,0.000000,0.000000
50%,1.000000,0.000000,0.000000,0.000000,1.000000
75%,1.000000,0.000000,0.000000,0.000000,2.000000
max,8.000000,349.000000,161.000000,380.000000,699.000000


## Data Cleaning

### Filtering aircrafts and events

We want to filter the dataset to include aircraft that the client is interested in an analysis of:
- inspect relevant columns
- figure out any reasonable imputations
- filter the dataset

In [8]:
# Inspect the columns we need to filter on
print(df['Aircraft.Category'].value_counts())
print('---')
print(df['Amateur.Built'].value_counts())

Aircraft.Category
Airplane             27617
Helicopter            3440
Glider                 508
Balloon                231
Gyrocraft              173
Weight-Shift           161
Powered Parachute       91
Ultralight              30
Unknown                 14
WSFT                     9
Powered-Lift             5
Blimp                    4
UNK                      2
Rocket                   1
ULTR                     1
Name: count, dtype: int64
---
Amateur.Built
No     80312
Yes     8475
Name: count, dtype: int64


In [9]:
# Inspect the Event.Date column
print(df['Event.Date'].head(10))

0    1948-10-24
1    1962-07-19
2    1974-08-30
3    1977-06-19
4    1979-08-02
5    1979-09-17
6    1981-08-01
7    1982-01-01
8    1982-01-01
9    1982-01-01
Name: Event.Date, dtype: object


In [10]:
# Convert Event.Date to datetime format
df['Event.Date'] = pd.to_datetime(df['Event.Date'], errors='coerce')

# Filter for airplanes only
df = df[df['Aircraft.Category'] == 'Airplane']

# Filter for professional builds only
df = df[df['Amateur.Built'] == 'No']

# Filter for events from 1983 onwards (max 40 year lifetime)
df = df[df['Event.Date'].dt.year >= 1983]

# Check how many rows remain
print(df.shape)

(21447, 31)


In [11]:
# Inspect the injury columns and Aircraft.damage
print(df[['Total.Fatal.Injuries', 'Total.Serious.Injuries', 
          'Total.Minor.Injuries', 'Total.Uninjured']].isnull().sum())
print('---')
print(df['Aircraft.damage'].value_counts())

Total.Fatal.Injuries      2750
Total.Serious.Injuries    2828
Total.Minor.Injuries      2544
Total.Uninjured            711
dtype: int64
---
Aircraft.damage
Substantial    16990
Destroyed       2316
Minor            817
Unknown           97
Name: count, dtype: int64


In [12]:
# Fill NaNs in injury columns with 0
df['Total.Fatal.Injuries'] = df['Total.Fatal.Injuries'].fillna(0)
df['Total.Serious.Injuries'] = df['Total.Serious.Injuries'].fillna(0)
df['Total.Minor.Injuries'] = df['Total.Minor.Injuries'].fillna(0)
df['Total.Uninjured'] = df['Total.Uninjured'].fillna(0)

# Estimate total passengers on each flight
df['Total.Passengers'] = (df['Total.Fatal.Injuries'] + 
                          df['Total.Serious.Injuries'] + 
                          df['Total.Minor.Injuries'] + 
                          df['Total.Uninjured'])

# Create fraction of passengers fatally or seriously injured
df['Fatal.Serious.Fraction'] = (df['Total.Fatal.Injuries'] + df['Total.Serious.Injuries']) / df['Total.Passengers']

# Create a binary column: was the aircraft destroyed?
df['Destroyed'] = (df['Aircraft.damage'] == 'Destroyed').astype(int)

# Check the new columns
print(df[['Total.Passengers', 'Fatal.Serious.Fraction', 'Destroyed']].head(10))

      Total.Passengers  Fatal.Serious.Fraction  Destroyed
4149             588.0                0.000000          0
4150             588.0                0.000000          0
4171               2.0                1.000000          1
4285               5.0                0.200000          0
5957             289.0                0.000000          0
5960              26.0                0.500000          1
6669              30.0                0.033333          0
6760             100.0                0.000000          0
6806               1.0                0.000000          0
7084               1.0                0.000000          0


In [13]:
# Handle division by zero - if total passengers is 0, set fraction to NaN
df['Fatal.Serious.Fraction'] = df['Fatal.Serious.Fraction'].where(df['Total.Passengers'] > 0, other=float('nan'))

# Check how many NaNs we have in the fraction column
print(df['Fatal.Serious.Fraction'].isnull().sum())
print(df['Fatal.Serious.Fraction'].describe())

904
count    20543.000000
mean         0.283971
std          0.431704
min          0.000000
25%          0.000000
50%          0.000000
75%          0.800000
max          1.000000
Name: Fatal.Serious.Fraction, dtype: float64


In [14]:
# Inspect the Make column
print(df['Make'].isnull().sum())
print('---')
print(df['Make'].value_counts().head(20))

3
---
Make
CESSNA                4867
PIPER                 2803
Cessna                2279
Piper                 1186
BOEING                1037
BEECH                 1018
Beech                  413
MOONEY                 238
Boeing                 227
CIRRUS DESIGN CORP     218
AIR TRACTOR INC        217
AIRBUS                 215
BELLANCA               158
AERONCA                149
MAULE                  144
Mooney                 125
EMBRAER                123
Air Tractor            117
LUSCOMBE                95
DEHAVILLAND             91
Name: count, dtype: int64


In [15]:
# Standardize Make column to title case and strip whitespace
df['Make'] = df['Make'].str.strip().str.title()

# Check the result
print(df['Make'].value_counts().head(20))

Make
Cessna                7146
Piper                 3989
Beech                 1431
Boeing                1264
Mooney                 363
Airbus                 243
Cirrus Design Corp     220
Air Tractor Inc        219
Bellanca               219
Maule                  215
Air Tractor            206
Aeronca                200
Champion               158
Embraer                153
Grumman                147
Luscombe               141
Cirrus                 137
Stinson                129
Mcdonnell Douglas      108
North American         106
Name: count, dtype: int64


In [16]:
# Keep only makes with 50 or more records
make_counts = df['Make'].value_counts()
valid_makes = make_counts[make_counts >= 50].index
df = df[df['Make'].isin(valid_makes)]

# Check how many rows and unique makes remain
print(df.shape)
print('---')
print(f"Number of unique makes: {df['Make'].nunique()}")

(17892, 34)
---
Number of unique makes: 36


In [17]:
# Inspect the Model column
print(df['Model'].isnull().sum())
print('---')
print(df['Model'].value_counts().head(20))

13
---
Model
172          769
737          403
152          316
182          304
172S         276
PA28         273
172N         249
SR22         240
180          213
A36          181
172M         180
150          179
PA-18-150    175
PA-28-140    169
172P         143
140          117
172R         109
170B         107
PA-28-180    105
PA-28-161    102
Name: count, dtype: int64


In [18]:
# Drop rows with NaN in Model column
df = df.dropna(subset=['Model'])

# Create a unique Make_Model identifier
df['Make_Model'] = df['Make'] + ' ' + df['Model']

# Check the result
print(df['Make_Model'].value_counts().head(10))

Make_Model
Cessna 172     769
Boeing 737     403
Cessna 152     316
Cessna 182     304
Cessna 172S    276
Piper PA28     273
Cessna 172N    249
Cessna 180     213
Cessna 172M    180
Cessna 150     179
Name: count, dtype: int64


In [19]:
# Inspect the other relevant columns
print(df['Engine.Type'].value_counts())
print('---')
print(df['Weather.Condition'].value_counts())
print('---')
print(df['Number.of.Engines'].value_counts())
print('---')
print(df['Purpose.of.flight'].value_counts())
print('---')
print(df['Broad.phase.of.flight'].value_counts())

Engine.Type
Reciprocating      12835
Turbo Prop           931
Turbo Fan            701
Unknown              105
Turbo Jet             71
Geared Turbofan       12
Turbo Shaft            9
UNK                    1
Name: count, dtype: int64
---
Weather.Condition
VMC    14295
IMC      905
Unk      186
UNK       76
Name: count, dtype: int64
---
Number.of.Engines
1.0    13222
2.0     2470
4.0       67
3.0       26
0.0        5
Name: count, dtype: int64
---
Purpose.of.flight
Personal                     9844
Instructional                2410
Aerial Application            724
Business                      409
Unknown                       303
Positioning                   269
Skydiving                     157
Aerial Observation            147
Other Work Use                121
Banner Tow                     86
Flight Test                    73
Ferry                          72
Executive/corporate            65
Glider Tow                     29
Public Aircraft - Federal      28
Public Aircraft  

In [20]:
# Clean Engine.Type - combine Unknown and UNK
df['Engine.Type'] = df['Engine.Type'].replace('UNK', 'Unknown')

# Clean Weather.Condition - combine Unk and UNK
df['Weather.Condition'] = df['Weather.Condition'].replace({'Unk': 'Unknown', 'UNK': 'Unknown'})

# Clean Number.of.Engines - replace 0 with NaN (not possible)
df['Number.of.Engines'] = df['Number.of.Engines'].replace(0.0, float('nan'))

# Clean Purpose.of.flight - merge duplicate categories
df['Purpose.of.flight'] = df['Purpose.of.flight'].replace({
    'Air Race/show': 'Air Race show',
    'Unknown': float('nan')
})

# Clean Broad.phase.of.flight - replace Unknown with NaN
df['Broad.phase.of.flight'] = df['Broad.phase.of.flight'].replace('Unknown', float('nan'))

# Verify
print(df['Engine.Type'].value_counts())
print('---')
print(df['Weather.Condition'].value_counts())

Engine.Type
Reciprocating      12835
Turbo Prop           931
Turbo Fan            701
Unknown              106
Turbo Jet             71
Geared Turbofan       12
Turbo Shaft            9
Name: count, dtype: int64
---
Weather.Condition
VMC        14295
IMC          905
Unknown      262
Name: count, dtype: int64


In [21]:
# Check non-null counts for all columns
print(df.isnull().sum().sort_values(ascending=False))

Schedule                  15740
Broad.phase.of.flight     15438
Air.carrier                9431
Airport.Code               6231
Airport.Name               6125
Report.Status              3785
Purpose.of.flight          3350
Engine.Type                3214
Weather.Condition          2417
Number.of.Engines          2094
Longitude                  1901
Latitude                   1898
Aircraft.damage            1064
Publication.Date            788
Fatal.Serious.Fraction      776
Injury.Severity             717
FAR.Description             344
Registration.Number         164
Location                      4
Country                       1
Total.Passengers              0
Total.Minor.Injuries          0
Destroyed                     0
Total.Uninjured               0
Event.Id                      0
Total.Serious.Injuries        0
Total.Fatal.Injuries          0
Investigation.Type            0
Amateur.Built                 0
Model                         0
Make                          0
Aircraft

In [22]:
# Drop columns with more than 50% NaN values
threshold = len(df) * 0.5
df = df.dropna(axis=1, thresh=threshold)

# Check remaining columns
print(df.shape)
print('---')
print(df.columns.tolist())

(17879, 32)
---
['Event.Id', 'Investigation.Type', 'Accident.Number', 'Event.Date', 'Location', 'Country', 'Latitude', 'Longitude', 'Airport.Code', 'Airport.Name', 'Injury.Severity', 'Aircraft.damage', 'Aircraft.Category', 'Registration.Number', 'Make', 'Model', 'Amateur.Built', 'Number.of.Engines', 'Engine.Type', 'FAR.Description', 'Purpose.of.flight', 'Total.Fatal.Injuries', 'Total.Serious.Injuries', 'Total.Minor.Injuries', 'Total.Uninjured', 'Weather.Condition', 'Report.Status', 'Publication.Date', 'Total.Passengers', 'Fatal.Serious.Fraction', 'Destroyed', 'Make_Model']


In [23]:
# Save the cleaned DataFrame to a CSV file
df.to_csv('data/cleaned_aviation_data.csv', index=False)

# Confirm it saved
print("Saved successfully!")
print(df.shape)

Saved successfully!
(17879, 32)


### Cleaning and constructing Key Measurables

Injuries and robustness to destruction are a key interest point for the client. Clean and impute relevant columns and then create derived fields that best quantifies what the client wishes to track. **Use commenting or markdown to explain any cleaning assumptions as well as any derived columns you create.**

**Construct metric for fatal/serious injuries**

*Hint:* Estimate the total number of passengers on each flight. The likelihood of serious / fatal injury can be estimated as a fraction from this.

**Aircraft.Damage**
- identify and execute any cleaning tasks
- construct a derived column tracking whether an aircraft was destroyed or not.

### Investigate the *Make* column
- Identify cleaning tasks here
- List cleaning tasks clearly in markdown
- Execute the cleaning tasks
- For your analysis, keep Makes with a reasonable number (you can put the threshold at 50 though lower could work as well)

### Inspect Model column
- Get rid of any NaNs.
- Inspect the column and counts for each model/make. Are model labels unique to each make?
- If not, create a derived column that is a unique identifier for a given plane type.

### Cleaning other columns
- there are other columns containing data that might be related to the outcome of an accident. We list a few here:
- Engine.Type
- Weather.Condition
- Number.of.Engines
- Purpose.of.flight
- Broad.phase.of.flight

Inspect and identify potential cleaning tasks in each of the above columns. Execute those cleaning tasks. 

**Note**: You do not necessarily need to impute or drop NaNs here.

### Column Removal
- inspect the dataframe and drop any columns that have too many NaNs

### Save DataFrame to csv
- its generally useful to save data to file/server after its in a sufficiently cleaned or intermediate state
- the data can then be loaded directly in another notebook for further analysis
- this helps keep your notebooks and workflow readable, clean and modularized